In [1]:
texts = ["i love this movie", "this film is great", "terrible movie", "worst film ever", "amazing story", "boring and bad"]
labels = [1, 1, 0, 0, 1, 0]  # 1=긍정, 0=부정

In [5]:
vocab = {}
for text in texts:
    for word in text.split():
        if word not in vocab:
            vocab[word] = len(vocab) + 1  # 1부터 시작
vocab

{'i': 1,
 'love': 2,
 'this': 3,
 'movie': 4,
 'film': 5,
 'is': 6,
 'great': 7,
 'terrible': 8,
 'worst': 9,
 'ever': 10,
 'amazing': 11,
 'story': 12,
 'boring': 13,
 'and': 14,
 'bad': 15}

In [6]:
def encode(text, vocab):
    return [vocab[word] for word in text.split()]

In [8]:
encoded = [encode(t, vocab) for t in texts]
encoded

[[1, 2, 3, 4], [3, 5, 6, 7], [8, 4], [9, 5, 10], [11, 12], [13, 14, 15]]

In [10]:
max_len = max(len(s) for s in encoded)
padded = [s + [0] * (max_len - len(s)) for s in encoded]
padded

[[1, 2, 3, 4],
 [3, 5, 6, 7],
 [8, 4, 0, 0],
 [9, 5, 10, 0],
 [11, 12, 0, 0],
 [13, 14, 15, 0]]

In [16]:
import torch.nn as nn

X = torch.tensor(padded)
y = torch.tensor(labels)

In [21]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.linear = nn.Linear(hidden_dim, output_dim)


    def forward(self, x):
        x = self.embedding(x)
        output, (hidden,cell) = self.lstm(x)
        x = self.linear(hidden[-1])
        return x

In [22]:
model = LSTMClassifier(len(vocab)+1, 16, 32, 1)

loss_fn = nn.BCEWithLogitsLoss()  # 이진분류용
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [23]:
for epoch in range(100):
    predict = model(X)
    losscal = loss_fn(predict.squeeze(), y.float())
    optimizer.zero_grad()
    losscal.backward()
    optimizer.step()

In [26]:
model.eval()
with torch.no_grad():
    predict = model(X)
    predicted = (predict.squeeze() > 0).long()
    print(predicted.tolist())
    print(labels)


[1, 1, 0, 0, 1, 0]
[1, 1, 0, 0, 1, 0]
